# Thu thập dữ liệu bất động sản TP. Hồ Chí Minh

Notebook này hiện tập trung chạy **NhaTot (Chợ Tốt)**. Phần Batdongsan.com.vn được giữ lại ở chế độ đọc response HTML offline và tạm hoãn vì website đang trả HTTP 403 cho request tự động.

## Cách chạy từ đầu

1. Chạy cell 2 để import thư viện và tạo đường dẫn.
2. Chạy cell 3 để nạp các hàm và cấu hình NhaTot.
3. Chạy cell 4 để nạp parser Batdongsan offline.
4. Chạy cell 6 để thu thập NhaTot. Cell này đọc checkpoint cũ và tiếp tục đến 5.000 tin.
5. Chạy cell 7 chỉ để xem trạng thái Batdongsan đang tạm hoãn.
6. Chạy cell 8 để kiểm tra số dòng, ID/URL trùng và file raw.

## Cấu trúc dữ liệu

- `data/raw/nhatot_raw.csv`: dữ liệu NhaTot và checkpoint.
- `data/raw/batdongsan_raw.csv`: dữ liệu Batdongsan nếu sau này có response hợp lệ.
- `data/raw/batdongsan_pages/`: nơi đặt các response HTML Batdongsan đã lưu hợp lệ để parse offline.
- `logs/`: log request và tiến trình.

Checkpoint được ghi sau mỗi 150 dòng. Không xóa file raw khi chạy lại.

In [18]:
# Bước 1: Import thư viện và thiết lập đường dẫn
from pathlib import Path
import json
import logging
import random
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

CURRENT_FOLDER = Path.cwd()
PROJECT_ROOT = (
    CURRENT_FOLDER.parent
    if CURRENT_FOLDER.name.lower() == 'source'
    else CURRENT_FOLDER
)
RAW_FOLDER = PROJECT_ROOT / 'data' / 'raw'
LOG_FOLDER = PROJECT_ROOT / 'logs'
RAW_FOLDER.mkdir(parents=True, exist_ok=True)
LOG_FOLDER.mkdir(parents=True, exist_ok=True)

TARGET_PER_SOURCE = 5000
CHECKPOINT_EVERY = 150
REQUEST_DELAY = (1.5, 3.0)
BLOCKED_STATUS_CODES = (403, 429)
LAST_REQUEST_BLOCKED = False
OUTPUT_COLUMNS = [
    'ma_tin', 'tieu_de', 'loai_hinh', 'gia', 'dien_tich',
    'so_phong_ngu', 'so_tang', 'huong_nha', 'tinh_thanh',
    'quan_huyen', 'phuong_xa', 'tinh_trang_phap_ly', 'ngay_dang',
    'ten_moi_gioi_san', 'nguon', 'khu_vuc', 'dia_diem', 'mo_ta',
    'nguoi_dang', 'thoi_gian_dang', 'so_phong', 'url'
]
NHATOT_API_URL = 'https://gateway.chotot.com/v1/public/ad-listing'
NHATOT_API_LIMIT = 10
NHATOT_API_OFFSET_PARAM = 'o'
NHATOT_API_PARAMS = {
    'limit': NHATOT_API_LIMIT,
    'protection_entitlement': 'true',
    'cg': 1000,
    'region_v2': 13000,
    'key_param_included': 'true',
    'video_count_included': 'true'
}
USER_AGENT = (
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 Chrome/125.0 Safari/537.36'
)

REQUEST_LOG_FILE = LOG_FOLDER / 'request_blocked.log'
request_logger = logging.getLogger('real_estate_crawler')
request_logger.setLevel(logging.WARNING)
request_logger.propagate = False
if not request_logger.handlers:
    log_handler = logging.FileHandler(REQUEST_LOG_FILE, encoding='utf-8')
    log_handler.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s'
    ))
    request_logger.addHandler(log_handler)

def tao_session():
    # Session chỉ dùng header tối thiểu cho request công khai.
    session = requests.Session()
    session.headers.update({
        'User-Agent': USER_AGENT,
        'Accept-Language': 'vi-VN,vi;q=0.9,en;q=0.8',
        'Accept': 'application/json, text/plain, */*',
        'Origin': 'https://www.nhatot.com',
        'Referer': 'https://www.nhatot.com/'
    })
    return session

def request_json(session, url, params=None):
    global LAST_REQUEST_BLOCKED
    LAST_REQUEST_BLOCKED = False
    time.sleep(random.uniform(*REQUEST_DELAY))
    try:
        response = session.get(url, params=params, timeout=25)
        if response.status_code in BLOCKED_STATUS_CODES:
            LAST_REQUEST_BLOCKED = True
            request_logger.warning(
                'api_blocked status=%s url=%s', response.status_code, response.url
            )
            print(f'API từ chối request ({response.status_code}): {response.url}')
            return None
        response.raise_for_status()
        return response.json()
    except (requests.RequestException, ValueError) as error:
        status_code = getattr(
            getattr(error, 'response', None), 'status_code', 'unknown'
        )
        request_logger.warning(
            'api_failed status=%s url=%s error=%s', status_code, url, error
        )
        print(f'Không đọc được API {url}: {error}')
        return None

def request_html(session, url):
    global LAST_REQUEST_BLOCKED
    LAST_REQUEST_BLOCKED = False
    time.sleep(random.uniform(*REQUEST_DELAY))
    try:
        response = session.get(url, timeout=25)
        if response.status_code in BLOCKED_STATUS_CODES:
            LAST_REQUEST_BLOCKED = True
            request_logger.warning(
                'blocked status=%s url=%s', response.status_code, url
            )
            print(f'Website từ chối request ({response.status_code}): {url}')
            return None
        response.raise_for_status()
        return response.text
    except requests.RequestException as error:
        status_code = getattr(error.response, 'status_code', 'unknown')
        request_logger.warning(
            'request_failed status=%s url=%s error=%s', status_code, url, error
        )
        print(f'Không tải được {url}: {error}')
        return None

def lay_text_tho(element, selectors):
    for selector in selectors:
        found = element.select_one(selector) if element else None
        if found:
            return found.get_text(' ', strip=False)
    return ''

def lay_attr_tho(element, selectors, attribute):
    for selector in selectors:
        found = element.select_one(selector) if element else None
        if found and found.get(attribute) is not None:
            return found.get(attribute)
    return ''

def json_ld_value_tho(soup, keys):
    for script in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(script.string or script.get_text())
            items = data if isinstance(data, list) else [data]
            for item in items:
                if isinstance(item, dict):
                    for key in keys:
                        if item.get(key) is not None:
                            return str(item[key])
        except (json.JSONDecodeError, TypeError):
            pass
    return ''

def tao_ban_ghi_tho(record):
    return {column: record.get(column, '') for column in OUTPUT_COLUMNS}

def record_identity(record):
    """Khóa dedup ổn định, ưu tiên ID rồi đến URL."""
    source = str(record.get('nguon', '')).strip()
    ad_id = str(record.get('ma_tin', '')).strip()
    url = str(record.get('url', '')).strip()
    return (source, f'id:{ad_id}') if ad_id else (source, f'url:{url}')

def doc_checkpoint(file_path):
    if not file_path.exists():
        return []
    try:
        raw_records = pd.read_csv(
            file_path, dtype=str, keep_default_na=False
        ).to_dict('records')
    except (OSError, pd.errors.ParserError):
        return []
    unique_records = []
    seen = set()
    for record in raw_records:
        identity = record_identity(record)
        if identity[1] in ('id:', 'url:') or identity in seen:
            continue
        seen.add(identity)
        unique_records.append(tao_ban_ghi_tho(record))
    return unique_records

def ghi_checkpoint(records, file_path):
    if records:
        pd.DataFrame(records, columns=OUTPUT_COLUMNS).to_csv(
            file_path, index=False, encoding='utf-8-sig'
        )

In [19]:
# Bước 4: Hàm lấy dữ liệu NhaTot từ API JSON

def tim_param(ad, param_id):
    for item in ad.get('params', []):
        if item.get('id') == param_id:
            return item.get('value', '')
    return ''

def tao_url_nhatot(ad):
    return (
        ad.get('url')
        or ad.get('web_url')
        or f"https://www.nhatot.com/{ad.get('ad_id', '')}"
    )

def parse_nhatot_api_ad(ad):
    shop = ad.get('shop') or {}
    seller_info = ad.get('seller_info') or {}
    legal_status = ad.get('property_legal_document', '')
    return tao_ban_ghi_tho({
        'ma_tin': ad.get('ad_id', ''),
        'tieu_de': ad.get('subject', ''),
        'loai_hinh': (
            tim_param(ad, 'house_type')
            or tim_param(ad, 'apartment_type')
            or ad.get('category_name', '')
        ),
        'gia': ad.get('price_string', ad.get('price', '')),
        'dien_tich': ad.get('size', ad.get('area', '')),
        'so_phong_ngu': ad.get('rooms', '') or tim_param(ad, 'rooms'),
        'so_tang': ad.get('floors', '') or tim_param(ad, 'floors'),
        'huong_nha': ad.get('direction', '') or tim_param(ad, 'direction'),
        'tinh_thanh': ad.get('region_name_v3', ad.get('region_name', '')),
        'quan_huyen': ad.get('area_name', ''),
        'phuong_xa': ad.get('ward_name_v3', ad.get('ward_name', '')),
        'tinh_trang_phap_ly': legal_status,
        'ngay_dang': ad.get('date', ''),
        'ten_moi_gioi_san': shop.get('name', ''),
        'nguon': 'nhatot',
        'khu_vuc': ad.get('area_name', ''),
        'dia_diem': shop.get('address', ''),
        'mo_ta': ad.get('body', ''),
        'nguoi_dang': seller_info.get('full_name') or ad.get('full_name', ''),
        'thoi_gian_dang': ad.get('date', ''),
        'so_phong': ad.get('rooms', '') or tim_param(ad, 'rooms'),
        'url': tao_url_nhatot(ad)
    })

def crawl_nhatot_api(
    target=TARGET_PER_SOURCE,
    checkpoint_path=None,
    max_pages=600
):
    checkpoint_path = checkpoint_path or RAW_FOLDER / 'nhatot_raw.csv'
    records = doc_checkpoint(checkpoint_path)
    seen_keys = {record_identity(record) for record in records}
    session = tao_session()
    api_offset = 0
    total = None
    pages_processed = 0
    checkpoint_bucket = len(records) // CHECKPOINT_EVERY

    for page_number in range(1, max_pages + 1):
        if len(records) >= target:
            break
        params = dict(NHATOT_API_PARAMS)
        params[NHATOT_API_OFFSET_PARAM] = api_offset
        response_data = request_json(session, NHATOT_API_URL, params=params)
        if response_data is None:
            print(
                f'NhaTot dừng: page={page_number}, offset={api_offset}, '
                f'tổng={len(records)}, checkpoint={checkpoint_path}'
            )
            break

        ads = response_data.get('ads') or []
        total = response_data.get('total', total)
        if not ads:
            print(
                f'NhaTot hết dữ liệu: page={page_number}, offset={api_offset}, '
                f'tổng={len(records)}, checkpoint={checkpoint_path}'
            )
            break

        pages_processed += 1
        new_count = 0
        duplicate_count = 0
        for ad in ads:
            record = parse_nhatot_api_ad(ad)
            identity = record_identity(record)
            if identity in seen_keys:
                duplicate_count += 1
                continue
            records.append(record)
            seen_keys.add(identity)
            new_count += 1
            if len(records) >= target:
                break

        current_bucket = len(records) // CHECKPOINT_EVERY
        if current_bucket > checkpoint_bucket:
            ghi_checkpoint(records, checkpoint_path)
            checkpoint_bucket = current_bucket
            checkpoint_status = f'đã lưu {len(records)} dòng'
        else:
            checkpoint_status = f'chưa đến mốc {CHECKPOINT_EVERY}'

        print(
            f'NhaTot: page={page_number}, offset={api_offset}, '
            f'listing nhận={len(ads)}, mới={new_count}, trùng={duplicate_count}, '
            f'tổng={len(records)}/{total or "?"}, checkpoint={checkpoint_status}'
        )

        api_offset += len(ads)
        if len(ads) < NHATOT_API_LIMIT:
            break

    ghi_checkpoint(records, checkpoint_path)
    print(
        f'NhaTot kết thúc: pages={pages_processed}, tổng={len(records)}, '
        f'checkpoint={checkpoint_path}'
    )
    return pd.DataFrame(records[:target], columns=OUTPUT_COLUMNS)

In [22]:
# Bước 6: Hàm đọc response HTML Batdongsan đã lưu hợp lệ
BATDONGSAN_RESPONSE_FOLDER = RAW_FOLDER / 'batdongsan_pages'


def tim_card_batdongsan(soup):
    return soup.select('.js__card.js__card-listing')


def tim_url_trong_card_batdongsan(card):
    link = card.select_one('a.js__product-link-for-product-id[href]')
    if not link:
        return ''
    return urljoin('https://batdongsan.com.vn', link.get('href', ''))


def parse_batdongsan_card(card):
    link = card.select_one('a.js__product-link-for-product-id[href]')
    url = tim_url_trong_card_batdongsan(card)
    ma_tin = (
        card.get('prid', '')
        or card.get('data-product-id', '')
        or (link.get('data-product-id', '') if link else '')
    )
    agent_name = lay_text_tho(card, ['.agent-name'])
    published_at = lay_text_tho(card, ['.re__card-published-info-published-at'])
    return tao_ban_ghi_tho({
        'ma_tin': ma_tin,
        'tieu_de': lay_text_tho(card, ['.js__card-title', '[product-title]']),
        'loai_hinh': '',
        'gia': lay_text_tho(card, ['.js__card-config-price']),
        'dien_tich': lay_text_tho(card, ['.js__card-config-area']),
        'so_phong_ngu': '',
        'so_tang': '',
        'huong_nha': '',
        'tinh_thanh': 'TP. Hồ Chí Minh',
        'quan_huyen': '',
        'phuong_xa': '',
        'tinh_trang_phap_ly': '',
        'ngay_dang': published_at,
        'ten_moi_gioi_san': agent_name,
        'nguon': 'batdongsan',
        'khu_vuc': lay_text_tho(card, ['.re__card-location']),
        'dia_diem': lay_text_tho(card, ['.re__card-location']),
        'mo_ta': lay_text_tho(card, ['.js__card-description']),
        'nguoi_dang': agent_name,
        'thoi_gian_dang': published_at,
        'so_phong': '',
        'url': url
    })


def crawl_batdongsan(
    target=TARGET_PER_SOURCE,
    checkpoint_path=None,
    response_folder=BATDONGSAN_RESPONSE_FOLDER
):
    """Parse local HTML responses; never requests Batdongsan or bypasses 403."""
    checkpoint_path = checkpoint_path or RAW_FOLDER / 'batdongsan_raw.csv'
    response_folder = Path(response_folder)
    response_files = sorted(response_folder.glob('*.html'))
    records = doc_checkpoint(checkpoint_path)
    seen_keys = {record_identity(record) for record in records}
    pages_processed = 0
    listings_received = 0
    duplicate_count = 0
    checkpoint_bucket = len(records) // CHECKPOINT_EVERY

    if not response_files:
        print(
            f'Batdongsan chưa có response HTML trong {response_folder}. '
            'Lưu các response hợp lệ vào đây rồi chạy lại cell này.'
        )
        ghi_checkpoint(records, checkpoint_path)
        return pd.DataFrame(records[:target], columns=OUTPUT_COLUMNS)

    for response_file in response_files:
        if len(records) >= target:
            break
        try:
            html = response_file.read_text(encoding='utf-8', errors='replace')
        except OSError as error:
            print(f'Không đọc được response {response_file}: {error}')
            continue

        pages_processed += 1
        cards = tim_card_batdongsan(BeautifulSoup(html, 'html.parser'))
        listings_received += len(cards)
        page_new = 0
        page_duplicate = 0
        for card in cards:
            if len(records) >= target:
                break
            record = parse_batdongsan_card(card)
            identity = record_identity(record)
            if not record.get('ma_tin') or not record.get('url'):
                continue
            if identity in seen_keys:
                duplicate_count += 1
                page_duplicate += 1
                continue
            records.append(record)
            seen_keys.add(identity)
            page_new += 1
            current_bucket = len(records) // CHECKPOINT_EVERY
            if current_bucket > checkpoint_bucket:
                ghi_checkpoint(records, checkpoint_path)
                checkpoint_bucket = current_bucket
                print(
                    f'Batdongsan checkpoint: {len(records)} dòng '
                    f'(file={response_file.name})'
                )

        print(
            f'Batdongsan response={response_file.name}, '
            f'listing nhận={len(cards)}, mới={page_new}, trùng={page_duplicate}, '
            f'tổng={len(records)}/{target}, tổng trùng={duplicate_count}'
        )

    ghi_checkpoint(records, checkpoint_path)
    print(
        f'Batdongsan kết thúc offline: response={pages_processed}, '
        f'listing nhận={listings_received}, mới={len(records)}, '
        f'trùng={duplicate_count}, checkpoint={checkpoint_path}'
    )
    return pd.DataFrame(records[:target], columns=OUTPUT_COLUMNS)

## Ghi chú nguồn dữ liệu

### NhaTot

NhaTot dùng endpoint JSON công khai `gateway.chotot.com/v1/public/ad-listing`. API phân trang bằng tham số `o`. Cấu hình bỏ bộ lọc `st='s,k'` vì bộ lọc này chỉ trả 521 tin; cấu hình hiện tại giữ `region_v2=13000` và trả khoảng 6.346 tin để có thể lấy 5.000 tin thật. Crawler tự đọc `data/raw/nhatot_raw.csv`, bỏ qua ID/URL đã có và ghi checkpoint sau mỗi 150 dòng.

### Batdongsan, tạm hoãn

Notebook không gửi request Batdongsan trong luồng chạy chính. Parser offline đọc các file `data/raw/batdongsan_pages/*.html` nếu sau này có response HTML hợp lệ. Không dùng CAPTCHA, cookie phiên, token cá nhân, proxy, fingerprint hoặc kỹ thuật vượt 403.

Các lỗi request NhaTot (`403`, `429`) được ghi vào `logs/request_blocked.log`.

In [14]:
# Bước 8: Chạy thu thập NhaTot qua API JSON
# Có thể thử target nhỏ trước, sau đó chạy target mặc định 5000.
nhatot_data = crawl_nhatot_api(target=TARGET_PER_SOURCE)
print(f'Đã thu thập NhaTot: {len(nhatot_data)} dòng')

NhaTot: page=1, offset=0, listing nhận=10, mới=10, trùng=0, tổng=531/6345, checkpoint=chưa đến mốc 150
NhaTot: page=2, offset=10, listing nhận=10, mới=8, trùng=2, tổng=539/6346, checkpoint=chưa đến mốc 150
NhaTot: page=3, offset=20, listing nhận=10, mới=10, trùng=0, tổng=549/6346, checkpoint=chưa đến mốc 150
NhaTot: page=4, offset=30, listing nhận=10, mới=8, trùng=2, tổng=557/6346, checkpoint=chưa đến mốc 150
NhaTot: page=5, offset=40, listing nhận=10, mới=10, trùng=0, tổng=567/6346, checkpoint=chưa đến mốc 150
NhaTot: page=6, offset=50, listing nhận=10, mới=8, trùng=2, tổng=575/6346, checkpoint=chưa đến mốc 150
NhaTot: page=7, offset=60, listing nhận=10, mới=10, trùng=0, tổng=585/6346, checkpoint=chưa đến mốc 150
NhaTot: page=8, offset=70, listing nhận=10, mới=10, trùng=0, tổng=595/6346, checkpoint=chưa đến mốc 150
NhaTot: page=9, offset=80, listing nhận=10, mới=9, trùng=1, tổng=604/6346, checkpoint=đã lưu 604 dòng
NhaTot: page=10, offset=90, listing nhận=10, mới=10, trùng=0, tổng=614

In [ ]:
# Bước 9: Batdongsan tạm hoãn
# Không tự đọc HTML hoặc gửi request. Chỉ chạy crawler offline khi đã có response hợp lệ.
batdongsan_data = pd.DataFrame(
    doc_checkpoint(RAW_FOLDER / 'batdongsan_raw.csv'),
    columns=OUTPUT_COLUMNS
)
print(
    f'Batdongsan đang tạm hoãn: {len(batdongsan_data)} dòng trong checkpoint. '
    'Chưa gửi request tới website.'
)

Batdongsan chưa có response HTML trong c:\Users\Nguyen Phu Cuong\Desktop\Final_Project_DAP_Nhom9_13\data\raw\batdongsan_pages. Lưu các response hợp lệ vào đây rồi chạy lại cell này.
Đã thu thập Batdongsan: 0 dòng


In [24]:
# Bước 10: Kiểm tra cuối cùng hai nguồn và toàn bộ file raw
nhatot_file = RAW_FOLDER / 'nhatot_raw.csv'
batdongsan_file = RAW_FOLDER / 'batdongsan_raw.csv'

nhatot_data = pd.DataFrame(
    doc_checkpoint(nhatot_file), columns=OUTPUT_COLUMNS
)
batdongsan_data = pd.DataFrame(
    doc_checkpoint(batdongsan_file), columns=OUTPUT_COLUMNS
)
all_raw_files = sorted(RAW_FOLDER.glob('*.csv'))

assert list(nhatot_data.columns) == OUTPUT_COLUMNS
assert list(batdongsan_data.columns) == OUTPUT_COLUMNS
assert set(nhatot_data['nguon'].unique()).issubset({'', 'nhatot'})
assert set(batdongsan_data['nguon'].unique()).issubset({'', 'batdongsan'})

all_records = pd.concat([nhatot_data, batdongsan_data], ignore_index=True)
all_keys = [record_identity(record) for record in all_records.to_dict('records')]
duplicate_key_count = len(all_keys) - len(set(all_keys))
duplicate_id_count = all_records['ma_tin'].astype(str).duplicated().sum()
duplicate_url_count = all_records['url'].astype(str).duplicated().sum()

print(f'NhaTot: {len(nhatot_data)} dòng (mục tiêu {TARGET_PER_SOURCE})')
print(f'Batdongsan: {len(batdongsan_data)} dòng (mục tiêu {TARGET_PER_SOURCE})')
print(f'Tổng cộng: {len(all_records)} dòng (mục tiêu {TARGET_PER_SOURCE * 2})')
print(f'Trùng khóa nguồn + ID/URL: {duplicate_key_count}')
print(f'Trùng ID: {duplicate_id_count}')
print(f'Trùng URL: {duplicate_url_count}')
print(f'Số file raw đã tạo: {len(all_raw_files)}')
print('File raw:')
for raw_file in all_raw_files:
    print(f' - {raw_file}')

if len(nhatot_data) < TARGET_PER_SOURCE:
    print('CẢNH BÁO: NhaTot chưa đạt 5.000 dòng dữ liệu thật.')
if len(batdongsan_data) < TARGET_PER_SOURCE:
    print(
        'CẢNH BÁO: Batdongsan chưa đạt 5.000 dòng; cần API/export/quyền '
        'truy cập chính thức vì route HTML đang trả 403.'
    )

NhaTot: 5000 dòng (mục tiêu 5000)
Batdongsan: 0 dòng (mục tiêu 5000)
Tổng cộng: 5000 dòng (mục tiêu 10000)
Trùng khóa nguồn + ID/URL: 0
Trùng ID: 0
Trùng URL: 0
Số file raw đã tạo: 1
File raw:
 - c:\Users\Nguyen Phu Cuong\Desktop\Final_Project_DAP_Nhom9_13\data\raw\nhatot_raw.csv
CẢNH BÁO: Batdongsan chưa đạt 5.000 dòng; cần API/export/quyền truy cập chính thức vì route HTML đang trả 403.
